# Detecção de Defeitos em Frutas e Hortaliças com YOLOv8 e YOLOv11

Este notebook apresenta um pipeline completo para detecção automática de defeitos em frutas e hortaliças usando visão computacional e redes neurais YOLOv8 e YOLOv11.

### Objetivo
Classificar e localizar frutas boas (**fresh**) e defeituosas (**rotten**), auxiliando em processos automatizados de inspeção de qualidade.

### Etapas do Pipeline
1. **Instalação de dependências**
2. **Download e organização do dataset**
3. **Configuração dos diretórios e labels**
4. **Treinamento com YOLOv8**
5. **Avaliação e detecção**

## 1.0 Instalação das Dependências

Nesta etapa, instalamos as bibliotecas necessárias para o projeto, incluindo:
- **Ultralytics**: Framework para YOLO.
- **OpenCV**: Processamento de imagens.
- **Matplotlib**: Visualização de dados.
- **OpenDatasets**: Download de datasets públicos.

In [ ]:
!pip install ultralytics opencv-python matplotlib opendatasets --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 52.5 MB/s eta 0:00:00


### 1.1 Importações e Verificação da Instalação

Aqui, importamos as bibliotecas instaladas e verificamos se estão funcionando corretamente.

In [ ]:
import os
from ultralytics import YOLO
import opendatasets as od
import shutil
import glob

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


## 2.0 Carregamento dos Datasets

Os datasets utilizados neste projeto são:
- **dataset_train**: Conjunto de treino principal.
- **dataset_train2**: Conjunto de treino adicional.
- **dataset_val**: Conjunto de validação.

Os datasets foram previamente armazenados no Google Drive e serão copiados para o ambiente do Colab.

In [ ]:
# Montar o Google Drive
from google.colab import drive
drive.mount('/content/drive/')

# Definir o caminho raiz no Drive
DRIVE_ROOT = '/content/drive/MyDrive/visao_computacional'

# Copiar os novos datasets do Drive para o /content/
print("Copiando datasets do Google Drive...")
!cp -r "{DRIVE_ROOT}/dataset_train" /content/
!cp -r "{DRIVE_ROOT}/dataset_train2" /content/
!cp -r "{DRIVE_ROOT}/dataset_val" /content/
print("Cópia concluída.")

Mounted at /content/drive/
Copiando datasets do Google Drive...
^C
cp: cannot stat '/content/drive/MyDrive/visao_computacional/dataset_train2': No such file or directory
cp: cannot stat '/content/drive/MyDrive/visao_computacional/dataset_val': No such file or directory
Cópia concluída.


## 3.0 Organização e Preparação do Dataset

Nesta etapa, unificamos os datasets de treino e preparamos a estrutura de diretórios e labels compatível com o YOLO.

### Estrutura de Labels
- **Rótulo 0**: Frutas boas (**fresh**)
- **Rótulo 1**: Frutas defeituosas (**rotten**)

### Estrutura de Diretórios
Os arquivos serão organizados da seguinte forma:
dataset_train/
├── fresh_apple/
├── fresh_banana/
├── fresh_orange/
├── rotten_apple/
├── rotten_banana/
├── rotten_orange/

dataset_train2/
├── fresh_apple/
├── fresh_banana/
├── fresh_orange/
├── rotten_apple/
├── rotten_banana/
├── rotten_orange/

dataset_val/
├── fresh_apple/
├── fresh_banana/
├── fresh_orange/
├── rotten_apple/
├── rotten_banana/
├── rotten_orange/

In [ ]:
import os
import shutil
import glob
import re # Usaremos regex para extrair o nome da classe

# Caminhos dos datasets de origem (copiados para /content/)
train_source_paths = ["/content/dataset_train", "/content/dataset_train2"]
val_source_path = "/content/dataset_val"

# Diretórios de destino padrão do YOLO
dest_base = "data"
os.makedirs(os.path.join(dest_base, "images", "train"), exist_ok=True)
os.makedirs(os.path.join(dest_base, "images", "val"), exist_ok=True)
os.makedirs(os.path.join(dest_base, "labels", "train"), exist_ok=True)
os.makedirs(os.path.join(dest_base, "labels", "val"), exist_ok=True)

def process_dataset(source_paths, subset):
    """
    Processa pastas de origem, atribui labels e copia para a estrutura YOLO.
    'source_paths' deve ser uma lista de caminhos.
    'subset' deve ser 'train' ou 'val'.
    """
    print(f"\nProcessando subset: {subset}")
    image_count = 0

    # Garantir que source_paths seja sempre uma lista
    if not isinstance(source_paths, list):
        source_paths = [source_paths]

    img_extensions = ["*.jpg", "*.png", "*.jpeg"]

    for source_path in source_paths:
        print(f"Lendo de: {source_path}")
        source_dataset_name = os.path.basename(source_path) # Ex: "dataset_train"

        # Encontrar todas as imagens dentro das subpastas (fresh_apple, rotten_banana, etc.)
        image_paths = []
        for ext in img_extensions:
            image_paths.extend(glob.glob(os.path.join(source_path, "*", ext)))

        for img_path in image_paths:
            try:
                # Ex: /content/dataset_train/fresh_apple/img1.jpg
                parent_folder_name = os.path.basename(os.path.dirname(img_path)) # Ex: "fresh_apple"

                # Definir classe com base no nome da pasta pai
                if "rotten" in parent_folder_name:
                    class_id = 1  # 1 para frutas podres
                elif "fresh" in parent_folder_name:
                    class_id = 0  # 0 para frutas frescas
                else:
                    print(f"  Aviso: Ignorando imagem {img_path}, pasta não contém 'fresh' ou 'rotten'.")
                    continue

                # --- Criar nome de arquivo único para evitar colisões ---
                base_name = os.path.basename(img_path)
                # Remover caracteres especiais do nome base para evitar problemas
                clean_base_name = re.sub(r"[^a-zA-Z0-9_.]", "", base_name)
                unique_name = f"{source_dataset_name}_{parent_folder_name}_{clean_base_name}"

                # --- Copiar imagem ---
                dest_img_path = os.path.join(dest_base, "images", subset, unique_name)
                shutil.copy(img_path, dest_img_path)

                # --- Criar rótulo (label) ---
                label_name = os.path.splitext(unique_name)[0] + ".txt"
                label_path = os.path.join(dest_base, "labels", subset, label_name)

                with open(label_path, "w") as f:
                    # class_id, x_center, y_center, width, height (normalizado)
                    f.write(f"{class_id} 0.5 0.5 1.0 1.0\n")

                image_count += 1

            except Exception as e:
                print(f"Erro ao processar {img_path}: {e}")

    print(f"Total de {image_count} imagens adicionadas ao subset {subset}.")
    return image_count

# --- Processar Imagens de Treino (dataset_train + dataset_train2) ---
num_train = process_dataset(train_source_paths, "train")

# --- Processar Imagens de Validação (dataset_val) ---
num_val = process_dataset(val_source_path, "val")

print(f"\n--- Resumo da Preparação ---")
print(f"Estrutura YOLO criada em: {dest_base}")
print(f"Total de imagens de treino: {num_train}")
print(f"Total de imagens de validação: {num_val}")


Processando subset: train
Lendo de: /content/dataset_train
Lendo de: /content/dataset_train2
Total de 15614 imagens adicionadas ao subset train.

Processando subset: val
Lendo de: /content/dataset_val
Total de 14414 imagens adicionadas ao subset val.

--- Resumo da Preparação ---
Estrutura YOLO criada em: data
Total de imagens de treino: 15614
Total de imagens de validação: 14414


## 4.0 Configuração do Arquivo `data.yaml`

O arquivo `data.yaml` informa ao YOLO onde estão localizados os dados e as classes. Ele contém:
- **train**: Caminho para as imagens de treino.
- **val**: Caminho para as imagens de validação.
- **nc**: Número de classes (neste caso, 2).
- **names**: Lista com os nomes das classes (`fresh` e `rotten`).

In [ ]:
yaml_content = """
train: data/images/train
val: data/images/val

nc: 2
names: ['fresh', 'rotten']
"""

with open("data.yaml", "w") as f:
    f.write(yaml_content)

print("Arquivo data.yaml criado com sucesso!")

Arquivo data.yaml criado com sucesso!


## 5.0 Treinamento com YOLOv8

Nesta etapa, realizamos o treinamento do modelo YOLOv8 com as seguintes configurações:
- **Épocas**: 300
- **Tamanho da imagem**: 640x640
- **Early Stopping**: Interrompe o treinamento se não houver melhora após 30 épocas.
- **Salvamento periódico**: Modelos intermediários são salvos a cada 10 épocas.

O modelo será salvo no Google Drive para facilitar o acesso posterior.

In [ ]:
from ultralytics import YOLO

# Carregar modelo base de segmentação
yolo_v8 = YOLO("yolov8n.pt")

# Treinamento avançado com early stopping e otimização de recursos
results_v8 = yolo_v8.train(
    data="data.yaml",
    epochs=300,
    imgsz=640,
    batch=-1,
    name="train_fruit_v8n",
    patience=30,
    save=True,
    save_period=10,
    plots=True,
    verbose=True,
    project="/content/drive/MyDrive/visao_computacional/runs"
)

Ultralytics 8.3.221 🚀 Python-3.12.12 torch-2.8.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=-1, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=300, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=fruit_defect_seg_longtrain, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=30, perspective=0.0, plots=True, pose

### 5.1 Treinamento com YOLOv11

O treinamento do YOLOv11 segue as mesmas configurações do YOLOv8, mas utiliza um modelo base diferente (`yolo11n.pt`).

In [ ]:
## 5.1 Treinamento YOLOv11n (Cópia da Célula 5.0)

from ultralytics import YOLO

yolo_v11 = YOLO("yolo11n.pt")

results_v11 = yolo_v11.train(
    data="data.yaml",
    epochs=300,
    imgsz=640,
    batch=-1,
    name="train_fruit_v11n",
    patience=30,
    save=True,
    save_period=10,
    plots=True,
    verbose=True,
    project="/content/drive/MyDrive/visao_computacional/runs"
)

## 6.0 Visualização das Métricas de Treinamento

Durante o treinamento, o YOLO gera gráficos que mostram a evolução das métricas ao longo das épocas. Nesta etapa, analisamos as seguintes métricas:
- **Loss**: Perda de treino e validação.
- **Precision**: Precisão do modelo.
- **Recall**: Revocação do modelo.
- **mAP@0.5**: Média de precisão para IoU ≥ 0.5.
- **mAP@0.5:0.95**: Média de precisão para IoU entre 0.5 e 0.95.

Os gráficos são gerados a partir dos arquivos `results.csv` de cada modelo.

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, HTML

def plot_training_metrics(run_dir_name, title):
    """
    Carrega o results.csv de uma pasta de treino específica e plota as métricas.
    """
    print("-" * 50)
    print(f"Gerando gráficos para: {title}")
    base_run_dir = "/content/drive/MyDrive/VisaoComputacional/runs"

    all_runs = sorted([d for d in os.listdir(base_run_dir) if d.startswith(run_dir_name)])
    if not all_runs:
        print(f"AVISO: Nenhuma pasta de run encontrada com o nome base: {run_dir_name}")
        print("Gráfico não pode ser gerado.")
        print("-" * 50)
        return None, None

    latest_run_folder = all_runs[-1]
    full_run_path = os.path.join(base_run_dir, latest_run_folder)
    results_csv = os.path.join(full_run_path, "results.csv")


    if not os.path.exists(results_csv):
        print(f"AVISO: O arquivo results.csv não foi encontrado em: {full_run_path}")
        print("Gráfico não pode ser gerado.")
        print("-" * 50)
        return None, None 

    df = pd.read_csv(results_csv)
    print(f"Métricas carregadas de: {results_csv}")

    df.columns = df.columns.str.strip()

    # Plotar métricas principais
    plt.figure(figsize=(15, 8))
    plt.subplot(2, 1, 1)
    plt.plot(df["epoch"], df["train/box_loss"], label="Perda de Treino (box_loss)")
    plt.plot(df["epoch"], df["val/box_loss"], label="Perda de Validação (box_loss)")
    plt.title(f"Perda (Loss) - {title}")
    plt.xlabel("Épocas")
    plt.ylabel("Perda")
    plt.legend()
    plt.grid(True)

    plt.subplot(2, 1, 2)
    plt.plot(df["epoch"], df["metrics/precision(B)"], label="Precisão (Precision)", marker='o', markersize=4)
    plt.plot(df["epoch"], df["metrics/recall(B)"], label="Revocação (Recall)", marker='o', markersize=4)
    plt.plot(df["epoch"], df["metrics/mAP50(B)"], label="mAP@0.5", marker='s', markersize=4, linestyle='--')
    plt.plot(df["epoch"], df["metrics/mAP50-95(B)"], label="mAP@0.5:0.95", marker='x', markersize=5, linestyle=':')
    plt.title(f"Métricas de Acurácia - {title}")
    plt.xlabel("Épocas")
    plt.ylabel("Pontuação")
    plt.legend()
    plt.grid(True)

    plt.tight_layout()
    plt.show()
    print("-" * 50)

    # Retorna o DataFrame e o caminho da pasta para uso posterior
    return df, full_run_path


In [ ]:
# 6.1 Gráficos do Treino YOLOv8
df_v8, v8_run_path = plot_training_metrics(
    run_dir_name="yolo_v8",
    title="YOLOv8 - Frutas Boas x Defeituosas"
)

In [ ]:
# 6.2 Gráficos do Treino YOLOv11
df_v11, v11_run_path = plot_training_metrics(
    run_dir_name="yolo_v11",
    title="YOLOv11n - Frutas Boas x Defeituosas"
)

## 7.0 Tabela Comparativa de Modelos

Consolidamos as métricas finais dos modelos treinados (YOLOv8 e YOLOv11) e as comparamos com um modelo base (FRCNN).

### Métricas Avaliadas
- **mAP@50**: Média de precisão para IoU ≥ 0.5.
- **mAP@50-95**: Média de precisão para IoU entre 0.5 e 0.95.
- **Precision**: Precisão do modelo.
- **Recall**: Revocação do modelo.
- **F1-score**: Calculado como `2 * (Precision * Recall) / (Precision + Recall)`.
- **Tempo de Inferência**: Tempo médio para processar uma imagem (ms/img).
- **Tamanho do Modelo**: Espaço ocupado pelo arquivo do modelo (`best.pt`).

### Objetivo
Identificar o modelo mais eficiente e preciso para a tarefa de detecção de defeitos em frutas e hortaliças.

In [ ]:
#import pandas as pd
#from IPython.display import display, HTML
#import os
#import json
#
## 7.1 Geração da Tabela de Comparação
#
## Caminho Raiz do Drive
#DRIVE_ROOT = '/content/drive/MyDrive/visao_computacional'
#
## Função para extrair métricas finais do DataFrame do YOLO
#def get_yolo_metrics(df, run_path):
#    if df is None or run_path is None:
#        return {"Error": "Dados do YOLO não encontrados"}
#
#    # Remover espaços das colunas
#    df.columns = df.columns.str.strip()
#
#    # Pegar as métricas da última época (ou a melhor)
#    best_epoch_data = df.iloc[-1]
#
#    precision = best_epoch_data["metrics/precision(B)"]
#    recall = best_epoch_data["metrics/recall(B)"]
#    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
#
#    # Tamanho do modelo
#    size_mb = "N/A"
#    weight_file = os.path.join(run_path, "weights", "best.pt")
#    if os.path.exists(weight_file):
#        size_mb = os.path.getsize(weight_file) / (1024 * 1024)
#
#    inference_ms = best_epoch_data["speed/inference"]
#
#    return {
#        "mAP@50": best_epoch_data["metrics/mAP50(B)"],
#        "mAP@50-95": best_epoch_data["metrics/mAP50-95(B)"],
#        "Precision": precision,
#        "Recall": recall,
#        "F1-score": f1,
#        "Tempo de inferência (ms/img)": inference_ms,
#        "Tamanho do modelo (MB)": size_mb,
#    }
#
## Carregar Métricas do FRCNN
#frcnn_metrics_path = os.path.join(DRIVE_ROOT, "runs/frcnn_training", "frcnn_metrics.json")
#frcnn_data = {}
#if os.path.exists(frcnn_metrics_path):
#    with open(frcnn_metrics_path, 'r') as f:
#        frcnn_data = json.load(f)
#    print("Métricas do FRCNN carregadas com sucesso.")
#else:
#    print(f"AVISO: Arquivo de métricas do FRCNN não encontrado em {frcnn_metrics_path}")
#    print("Preenchendo com N/A.")
#
## Carregar Métricas do YOLO
## (df_v8 e df_v11 foram definidos na célula 6.1 e 6.2)
#try:
#    yolo_v8_data = get_yolo_metrics(df_v8, v8_run_path)
#    yolo_v11_data = get_yolo_metrics(df_v11, v11_run_path)
#    print("Métricas do YOLO v8 e v11 carregadas com sucesso.")
#except NameError:
#    print("AVISO: df_v8 ou df_v11 não definidos. Execute a célula 6.0 primeiro.")
#    yolo_v8_data = {}
#    yolo_v11_data = {}
#
#
## Construir o Dicionário de Dados para a Tabela
## Usamos .get(chave, "N/A") para evitar erros se uma chave faltar
#data = {
#    "Critério": [
#        "mAP@50", "mAP@50-95", "Precision", "Recall", "F1-score",
#        "Tempo de inferência (ms/img)", "Tamanho do modelo (MB)",
#
#        # "Estabilidade (oscilações)", "Overfitting (gap treino/val)",
#        # "Eficiência", "Generalização"
#    ],
#    "YOLOv8": [
#        yolo_v8_data.get("mAP@50"), yolo_v8_data.get("mAP@50-95"),
#        yolo_v8_data.get("Precision"), yolo_v8_data.get("Recall"),
#        yolo_v8_data.get("F1-score"), yolo_v8_data.get("Tempo de inferência (ms/img)"),
#        yolo_v8_data.get("Tamanho do modelo (MB)"),
#        # 0.0057, 0.0277, 249.0, 0.2628 # Valores manuais da imagem
#    ],
#    "YOLOv11": [
#        yolo_v11_data.get("mAP@50"), yolo_v11_data.get("mAP@50-95"),
#        yolo_v11_data.get("Precision"), yolo_v11_data.get("Recall"),
#        yolo_v11_data.get("F1-score"), yolo_v11_data.get("Tempo de inferência (ms/img)"),
#        yolo_v11_data.get("Tamanho do modelo (MB)"),
#        # 0.0061, 0.0652, 341.0, 0.244 # Valores manuais da imagem
#    ],
#    "FRCNN": [
#        frcnn_data.get("mAP@50"), frcnn_data.get("mAP@50-95"),
#        frcnn_data.get("Precision"), frcnn_data.get("Recall"),
#        frcnn_data.get("F1-score"), frcnn_data.get("Tempo de inferência (ms/img)"),
#        frcnn_data.get("Tamanho do modelo (MB)"),
#        # 0.0018, -0.003, "N/A", 0.1864 # Valores manuais da imagem
#    ]
#}
#
## Criar o DataFrame
#df_comparacao = pd.DataFrame(data)
#df_comparacao = df_comparacao.set_index("Critério")
#
## Exibir a tabela formatada
#print("\n" + "="*50)
#print("       TABELA COMPARATIVA DE DESEMPENHO DE MODELOS")
#print("="*50)
#
## Formatar para exibir com 4 casas decimais onde for float
#display(df_comparacao.style.format(lambda x: f"{x:.4f}" if isinstance(x, float) else x, na_rep="N/A"))